In [1]:
import os
import shutil
import random
import csv
import glob

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================
ROOT_DIR = "./Brats_Scan"
SOURCE_SUB = "Train-Val"
DEST_SUB = "Test"
SPLIT_PERCENTAGE = 0.10  # 10%

In [3]:
def split_and_move():
    source_path = os.path.join(ROOT_DIR, SOURCE_SUB)
    dest_path = os.path.join(ROOT_DIR, DEST_SUB)

    # 1. Validation
    if not os.path.exists(source_path):
        print(f"Error: Source folder not found at {source_path}")
        return
    
    if not os.path.exists(dest_path):
        os.makedirs(dest_path)
        print(f"Created destination folder: {dest_path}")

    # 2. Get list of Patient Folders (exclude files)
    all_items = os.listdir(source_path)
    patient_folders = [f for f in all_items if os.path.isdir(os.path.join(source_path, f))]
    
    total_count = len(patient_folders)
    move_count = int(total_count * SPLIT_PERCENTAGE)
    
    if total_count == 0:
        print("No folders found to move.")
        return

    print(f"Found {total_count} folders.")
    print(f"Selecting {move_count} random folders (10%) to move to '{DEST_SUB}'...")

    # 3. Random Sampling
    folders_to_move = random.sample(patient_folders, move_count)

    # 4. Handle Log File Transfer
    # We try to find the CSV log to preserve metadata (Slice ID, Tumor Found, etc.)
    # We look for any CSV file containing 'dataset_log' in the source
    source_csv_candidates = glob.glob(os.path.join(source_path, "*dataset_log*.csv"))
    
    # Data structures for logs
    header = []
    source_rows = []    # Will keep remaining rows
    moved_rows = []     # Will store rows for the new log
    log_found = False
    source_csv_path = None

    if source_csv_candidates:
        source_csv_path = source_csv_candidates[0]
        print(f"Found log file: {os.path.basename(source_csv_path)}")
        
        try:
            with open(source_csv_path, 'r', newline='') as f:
                reader = csv.reader(f)
                header = next(reader, None) # Read header
                if header:
                    # Read all rows into memory
                    all_rows = list(reader)
                    
                    # Identify index of 'Patient_ID' to match folders
                    try:
                        # Assuming 'Patient_ID' is in header, case insensitive
                        # or just fallback to first column
                        id_idx = 0
                        for i, h in enumerate(header):
                            if 'patient' in h.lower() and 'id' in h.lower():
                                id_idx = i
                                break
                        
                        # Filter rows
                        files_set = set(folders_to_move)
                        for row in all_rows:
                            if row and len(row) > id_idx:
                                p_id = row[id_idx]
                                if p_id in files_set:
                                    # This row belongs to a moved folder
                                    # Update 'Output_Path' in the row to new location
                                    # Assuming 'Output_Path' is the last column or we verify header
                                    # For simplicity, we just move the row. 
                                    moved_rows.append(row)
                                else:
                                    source_rows.append(row)
                        log_found = True
                        
                    except Exception as e:
                        print(f"Warning: Could not parse CSV rows correctly. {e}")
        except Exception as e:
            print(f"Warning: Error reading log file. {e}")

    # If no log found, we create a basic header for the new log
    if not log_found:
        header = ['Patient_ID', 'Moved_From']
        # Create basic rows for moved files since we lack original metadata
        moved_rows = [[f, SOURCE_SUB] for f in folders_to_move]

    # 5. Perform the Physical Move
    print("Moving files...")
    for folder_name in folders_to_move:
        src = os.path.join(source_path, folder_name)
        dst = os.path.join(dest_path, folder_name)
        
        try:
            shutil.move(src, dst)
        except Exception as e:
            print(f"Error moving {folder_name}: {e}")

    # 6. Write New Log (Test)
    test_log_path = os.path.join(dest_path, "test_dataset_log.csv")
    with open(test_log_path, 'w', newline='') as f:
        writer = csv.writer(f)
        if header:
            writer.writerow(header)
        writer.writerows(moved_rows)
    
    print(f"Created new log: {test_log_path}")

    # 7. Update Old Log (Train-Val) - Optional but recommended to keep it clean
    if log_found and source_csv_path:
        with open(source_csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            if header:
                writer.writerow(header)
            writer.writerows(source_rows)
        print(f"Updated original log (removed moved entries): {os.path.basename(source_csv_path)}")

    print("-" * 30)
    print("Operation Complete.")

if __name__ == "__main__":
    split_and_move()

Found 1251 folders.
Selecting 125 random folders (10%) to move to 'Test'...
Found log file: train_dataset_log.csv
Moving files...
Created new log: ./Brats_Scan\Test\test_dataset_log.csv
Updated original log (removed moved entries): train_dataset_log.csv
------------------------------
Operation Complete.
